In [2]:
import os
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
from sklearn.utils import resample
import numpy as np

os.chdir('/Users/fogellmcmuffin/Documents/thesis/_replication/')    # Working dir

pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', None)

vintage_trim = pd.read_csv('cleaned_data/vintage_trim.csv')

mean_spf_trim = pd.read_csv('cleaned_data/mean_spf_trim.csv')
ind_spf_trim = pd.read_csv('cleaned_data/ind_spf_trim.csv')

mean_spf_trim['DATE'] = pd.to_datetime(mean_spf_trim['DATE'])
ind_spf_trim['DATE'] = pd.to_datetime(ind_spf_trim['DATE'])

ind_spf_trim = ind_spf_trim.dropna(subset='r_t1')

###############################################
                ## ESTIMATION ##
###############################################

### OLS ###
def OLS(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        revisions = df[f'r_t{j}']
        revisions = sm.add_constant(revisions)
        errors = df[f'e_t{j}']
        initial = sm.OLS(errors, revisions).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=None))
    return regs


### PLD OLS ###
def OLS_PLD(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        revisions = df[f'r_t{j}']
        revisions = sm.add_constant(revisions)
        errors = df[f'e_t{j}']
        regs.append(sm.OLS(errors, revisions).fit(cov_type='cluster', cov_kwds = {"groups":df['ID']}))
    return regs


### ID FE ###
def ID_FE(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['ID'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        regs.append(sm.OLS(y, x).fit(cov_type='cluster', cov_kwds = {"groups":df['ID']}))
    return regs


### AR Estimates ###
def AR(df, end_date):
    df = df.loc[(df['DATE'] >= '1965-06-30') & (df['DATE'] <= end_date)]  # Filter data
    grwth_t = df['t3']
    reg = AutoReg(grwth_t, 1).fit()
    return reg


### Parameter Estimates ###
def params(ols, pldols, fe, ar1):
    params = []
    params.append(ols / (1 + ols))
    params.append(1 / (1 + ols))
    params.append((-((2 * pldols) + 1) + np.sqrt(((2 * pldols) + 1)**2 - 4 * (pldols + (pldols * ar1**2) + 1) * pldols)) / (2 * (pldols + (pldols * ar1**2) + 1)))
    params.append((-((2 * fe) + 1) + np.sqrt(((2 * fe) + 1)**2 - 4 * (fe + (fe * ar1**2) + 1) * fe)) / (2 * (fe + (fe * ar1**2) + 1)))
    return params


def compute_regs(date, mean, ind):
    mean_regs = OLS(mean, f'{date}')
    ind_regs = OLS_PLD(ind, f'{date}')
    ind_regs_fe = ID_FE(ind, f'{date}')
    ar_1 = AR(vintage_trim, f'{date}')
    regs = [mean_regs, ind_regs, ind_regs_fe, ar_1]
    parameters = params(mean_regs[3].params[1], ind_regs[3].params[1], ind_regs_fe[3].params[1], ar_1.roots)
    return regs, parameters

regs_2020, parameters_2020 = compute_regs('2020-06-30', mean_spf_trim, ind_spf_trim)
%store regs_2020
%store parameters_2020

regs_2022, parameters_2022 = compute_regs('2022-12-31', mean_spf_trim, ind_spf_trim)
%store regs_2022
%store parameters_2022

/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Stored 'regs_2020' (list)
Stored 'parameters_2020' (list)


/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Stored 'regs_2022' (list)
Stored 'parameters_2022' (list)


In [3]:
regs_2020[0][3].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   e_t3   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.102
Date:                Fri, 03 May 2024   Prob (F-statistic):              0.295
Time:                        16:25:20   Log-Likelihood:                 486.11
No. Observations:                 155   AIC:                            -968.2
Df Residuals:                     153   BIC:                            -962.1
Df Model:                           1                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0023      0.001     -1.767      0.079      -0.005       0.000
r_t3           0.1936      0.184      1.050      0.295      -0.171       0.558
==============================================================================
Omnibus:                       11.153   Durbin-Watson:                   0.625
Prob(Omnibus):                  0.004   Jarque-Bera (JB):               16.683
Skew:                          -0.390   Prob(JB):                     0.000238
Kurtosis:                       4.405   Cond. No.                         220.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using None lags and without small sample correction
"""

In [6]:
parameters_2020[3]

array([0.5570091])